# NeuralMath — 神经网络的数学剖析

本文通过一个简单的分类问题，对人工神经网络作数学层面的入门介绍。线性分类器自然地引出人工神经元的概念；随后我们讨论权重、偏置、激活函数，以及如何把多个神经元组合成一个小型网络。训练过程通过损失函数、导数、梯度下降和反向传播来说明。最后，我们直接用 Python 实现该模型，使方程与算法之间的对应关系清晰可见。本文的目标是说明：解析几何、线性代数和微积分中的熟悉概念，可以为理解神经网络的基本机制提供一个易于进入的数学基础。

**NeuralMath.org** · *A anatomia matemática de uma rede neural*

变量名和函数名保留为英文；说明文字使用本页面的语言。


## 1. 构建一个小型神经网络

变量名和函数名保留为英文；说明文字使用本页面的语言。

$$\mathbf{z}^{(1)}=W^{(1)}\mathbf{x}+\mathbf{b}^{(1)},\qquad \mathbf{h}=\operatorname{ReLU}(\mathbf{z}^{(1)}),$$

$$z^{(2)}=W^{(2)}\mathbf{h}+b^{(2)},\qquad \widehat y=\sigma(z^{(2)}).$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def relu(s):
    return np.maximum(0, s)


def relu_derivative(s):
    return (s > 0).astype(float)


def sigmoid(s):
    return 1 / (1 + np.exp(-s))


def base_boundary(m):
    return (
        6.68
        + 0.74 * np.exp(-((m - 155) / 20) ** 2)
        + 1.50 / (1 + np.exp(-(m - 202) / 5))
    )


## 2. 问题的数学表示

为了理解神经网络如何工作，我们从一个有意简化的问题开始：区分苹果和橙子。水果具有许多可用于这项任务的特征，例如颜色、质量、直径、纹理和气味。不过，一个初始数学模型并不需要立即包含全部复杂性。我们只使用两个容易测量的量：以克为单位的质量，以及以厘米为单位的直径。

$$x_j^{\ast}=\frac{x_j-\mu_j}{\sigma_j}.$$


In [ ]:
def build_dataset():
    masses_1 = np.linspace(118, 212, 16)
    masses_2 = masses_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7,
    ])
    k = np.arange(16)

    apple_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) + 0.22 + 0.05 * np.sin(0.8 * k),
    ])
    apple_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) + 0.58 + 0.06 * np.cos(0.45 * k),
    ])
    orange_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) - (0.22 + 0.04 * np.cos(0.7 * k)),
    ])
    orange_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) - (0.55 + 0.05 * np.sin(0.5 * k)),
    ])

    X_train = np.vstack([apple_1, apple_2, orange_1, orange_2])
    y_train = np.array([1.0] * 32 + [0.0] * 32)

    X_test = np.array([
        [121.0, 7.20], [135.0, 7.45], [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86], [189.0, 6.48], [208.0, 7.55],
    ])
    y_test = np.array([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])
    return X_train, y_train, X_test, y_test


X_train, y_train, X_test, y_test = build_dataset()
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
Xn = (X_train - mean) / std

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('mean:', mean)
print('std:', std)


## 3. 神经网络如何学习？

权重和偏置并不是预先知道的。它们必须利用那些已经知道正确答案的样本来调整。这个过程称为 训练 ，可以概括为以下循环： \[\boxed{\text{预测}}
\longrightarrow
\boxed{\text{测量误差}}
\longrightarrow
\boxed{\text{调整参数}}
\longrightarrow
\boxed{\text{重复}}.\]


In [ ]:
rng = np.random.default_rng(42)
W1 = 0.1 * rng.standard_normal((12, 2))
b1 = np.zeros(12)
W2 = 0.1 * rng.standard_normal(12)
b2 = 0.0

learning_rate = 0.05
n_epochs = 6000
history = []

for epoch in range(n_epochs):
    for i in rng.permutation(len(Xn)):
        x = Xn[i]
        target = y_train[i]

        z1 = W1 @ x + b1
        h = relu(z1)
        z2 = W2 @ h + b2
        y_hat = sigmoid(z2)

        delta2 = (y_hat - target) * y_hat * (1 - y_hat)
        dW2 = delta2 * h
        db2 = delta2

        delta1 = (delta2 * W2) * relu_derivative(z1)
        dW1 = np.outer(delta1, x)
        db1 = delta1

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

    errors = []
    for x, target in zip(Xn, y_train):
        h = relu(W1 @ x + b1)
        y_hat = sigmoid(W2 @ h + b2)
        errors.append(0.5 * (target - y_hat) ** 2)
    history.append(float(np.mean(errors)))

print(f"E_final: {history[-1]:.10f}")


## 4. 从数学到 Python

现在，我们从方程转向一个可以实际运行的程序。目的不是使用专门的神经网络库，而是直接利用 NumPy 实现本文前面建立的数学运算 。这样，我们可以逐行看到线性组合、激活函数、损失计算、反向传播和梯度下降如何出现在代码中。


In [ ]:
def predict(X):
    X_scaled = (X - mean) / std
    outputs = []
    for x in X_scaled:
        h = relu(W1 @ x + b1)
        outputs.append(sigmoid(W2 @ h + b2))
    return np.array(outputs)


pred_train = predict(X_train)
pred_test = predict(X_test)

print(f"acc_train: {np.mean((pred_train >= 0.5) == y_train):.4f}")
print(f"acc_test: {np.mean((pred_test >= 0.5) == y_test):.4f}")
print('y_hat_test:', np.round(pred_test, 6))

masses = np.linspace(116, 214, 500)
diameters = np.linspace(6.0, 8.85, 500)
M, D = np.meshgrid(masses, diameters)
grid = np.column_stack([M.ravel(), D.ravel()])
P = predict(grid).reshape(M.shape)

fig, ax = plt.subplots(figsize=(9.2, 6.3))
ax.contourf(
    M, D, P,
    levels=[0.0, 0.25, 0.50, 0.75, 1.0],
    colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
    alpha=0.95,
)
ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'], linewidths=2.2)

mask = y_train == 1.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='o', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='苹果 - 训练')
mask = y_train == 0.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='s', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='橙子 - 训练')
mask = y_test == 1.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='^', s=92,
           color='#1f77b4', label='苹果 - 测试')
mask = y_test == 0.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='x', s=110,
           color='#ff7f0e', linewidths=2.0, label='橙子 - 测试')

ax.set_xlim(116, 214)
ax.set_ylim(6.0, 8.85)
ax.set_xlabel('质量 (g)')
ax.set_ylabel('直径 (cm)')
ax.set_title('神经网络训练后的输出', fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)
fig.tight_layout()
plt.show()


## 5. NeuralMath

我们从一条分隔两组简单点集的直线出发，最后得到一个更丰富的计算例子：通过自动调整网络参数，一条弯曲的边界从数据中形成。数学中并没有加入任何神秘的成分；实现只是把仿射组合、激活函数、导数、矩阵乘法和梯度下降更新这些基本运算，以更大的规模和更高的速度组织起来并反复执行 。
